In [1]:
%load_ext autoreload
%autoreload 2

In [3]:
import pandas as pd
import numpy as np
import json
import random
from src.utils import df_features_to_numpy, df_target_to_numpy, run_experiment_pytorch_clasificacion, run_experiment_pytorch_regresion, permutation_importance_pytorch
from sklearn.preprocessing import StandardScaler
from src.splitting import split_estratificado_autor

# Cargo los datasets

In [ ]:
#TODO: ver si sacar
chunks_test = pd.read_json("set_test_1.json", lines=True, chunksize=100_000)

chunk_list = []
for chunk in chunks_test:
    chunk_list.append(chunk)

df_test = pd.concat(chunk_list, ignore_index=True)

In [ ]:
chunks_val = pd.read_json("set_val_1.json", lines=True, chunksize=100_000)

chunk_list = []
for chunk in chunks_val:
    chunk_list.append(chunk)

df_val = pd.concat(chunk_list, ignore_index=True)

In [ ]:
chunks_train = pd.read_json("set_train_1.json", lines=True, chunksize=100_000)

chunk_list = []
for chunk in chunks_train:
    chunk_list.append(chunk)

df_train = pd.concat(chunk_list, ignore_index=True)

# Preparacion de los datos

In [10]:
generos = ["fiction", "non fiction", "fantasy", "science fiction", "mystery", "thriller", "romance", "horror", "historical", "young adult", "children", "adventure", "contemporary", "dystopian", "paranormal", "urban", "crime", "biography", "memoir", "self help", "history", "psychology", "poetry", "classics", "graphic novel", "short stories", "humor", "religion"]
features = ["is_series", "publication_year", "num_pages", 'format_group_AUDIO', 'format_group_COMIC', 'format_group_EBOOK', 'format_group_HARDCOVER', 'format_group_OTHER', 'format_group_PAPERBACK'] + generos #'format_group_UNKNOWN' la saco por el one hot en regresion

#Me quedo solo con las features necesarias para el modelo base y las estandarizo
dfs = [df_train, df_val, df_test]
y_train_regresion, y_val_regresion, y_test_regresion = df_target_to_numpy("average_rating", dfs)
y_train_clasificacion, y_val_clasificacion, y_test_clasificacion = df_target_to_numpy("estrellas", dfs)
X_train, X_val, X_test = df_features_to_numpy(features, dfs)

#estandarizo los datos
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val) #escala X_val con los valores sacados de X_train
#TODO: escalar el X_test

In [ ]:
from sklearn.neighbors import NearestNeighbors

modelo = NearestNeighbors(n_neighbors = 20, metric = "cosine")
modelo.fit(X_train_scaled)

def recomendar(modelo, X, indice_libro, n_recomendaciones = 10):
    distancias, indices = modelo.kneighbors(X[indice_libro].reshape(1, -1), n_neighbors = n_recomendaciones + 1)
    return indices[0][1:], distancias[0][1:]

In [16]:
idx = df_train[df_train["title"] == "The Book Thief"].index[0]
print(idx)

6016


TODO: que la persona pueda poner tb el autor, porq un mismo titulo puede correpsonder a diferentes libros

Va a funcionar mal porq todavia no puse los embeddings, pero asi seria aprox el codigo :)

In [17]:
recomendados, distancias = recomendar(modelo, X_train_scaled, 6016, 10)

print("\nRecomendaciones:")
for idx, dist in zip(recomendados, distancias):
    print(f"{df_train.iloc[idx]['title']}  (distancia = {dist:.4f})")


Recomendaciones:
Refugee (Bio of a Space Tyrant, #1)  (distancia = 0.1565)
Demon (Gaea, #3)  (distancia = 0.1654)
Steel Gauntlet (Starfist, #3)  (distancia = 0.1753)
Doctor Who: Relative Dementias  (distancia = 0.1880)
Courageous (The Lost Fleet, #3)  (distancia = 0.2055)
Life Probe  (distancia = 0.2153)
Mere Mortals (Star Trek: Destiny #2)  (distancia = 0.2199)
Courageous (The Lost Fleet, #3)  (distancia = 0.2240)
Transcendence (Heritage Universe 3)  (distancia = 0.2241)
Empire of Blood (Richard Blade, #23)  (distancia = 0.2244)
